In [1]:
!pip install -U langchain langchain-openai langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google

In [2]:
from google.colab import userdata
import os

# os.environ에 담는 이유
# 우리가 가져다 쓰는 각 llm provider사들의 라이브러리 함수들이 첫 시작으로 자식의 key값을 os.environ에서 찾는 것이 첫 시작이기 때문

# os.environ['LANGSMITH_TRACING'] = 'true'
# os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
# os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
# os.environ['LANGSMITH_PROJECT'] = "skn26_langchain"

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [7]:
# csv읽고 df로 가져오기(해당 컬럼만)
import pandas as pd

df = pd.read_csv('menu.csv')

menu_df = df[['name', 'description']].copy()

menu_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         2008 non-null   object
 1   description  980 non-null    object
dtypes: object(2)
memory usage: 31.5+ KB


In [8]:
# 문자열로 변환하고 공백 제거한다.
# 모든 null은 그대로 공백으로 대치하여 유지.

menu_df['name'] = menu_df['name'].astype(str).str.strip()
menu_df['description'] = menu_df['description'].astype(str).str.strip()

# 'nan' 문자열 처리 (astype(str) 하면 NaN → 'nan' 됨)
menu_df['name'] = menu_df['name'].replace('nan', '')
menu_df['description'] = menu_df['description'].replace('nan', '')

# 둘 다 null(=공백)인 경우 → 그대로 유지 (이미 '' 상태)
# 따로 삭제 안 함
print(menu_df.head(12))

                   name                                       description
0                  회전초밥                                                  
1                  장어초밥                                                  
2          프레지에 1호 홀케이크                                                  
3                  프레지에                                                  
4               페르디셔 압펠                                                  
5         루이보스 스트로베리 크림                                                  
6               바바리안 민트                                                  
7                 아메리카노                                                  
8                   마카롱                                                  
9   [패스오더전용] 100원 아메리카노  패스오더 앱 설치하면, 최소 주문 금액 없이 100원에 커피 주문+1,500원 쿠폰까지
10                 카페라떼                                                  
11                 카푸치노                                                  


In [13]:
# len(food_df)

menu_df = pd.read_csv('menu.csv')
menu = menu_df['name'].tolist()
len(menu)

2008

In [28]:
# OpenAI 프롬프트용 입력 만들기
menu_df['prompt_input'] = (
    'menu_name: ' + menu_df['name'].fillna('').astype(str) +
    '\n설명: ' + menu_df['description'].fillna('').astype(str)
)

In [29]:
menu_df['prompt_input'][:3]

,prompt_input
0,menu_name: 회전초밥\n설명:
1,menu_name: 장어초밥\n설명:
2,menu_name: 프레지에 1호 홀케이크\n설명:


In [30]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

food_prompt = PromptTemplate.from_template(
    """
너는 메뉴 분류를 잘하는 봇이야.

규칙:
- 주어진 메뉴명과 설명을 보고 대표 메뉴를 한 단어 또는 매우 짧은 명사형으로 정의해.
- 예시:
  - 치즈 돈가스 -> 돈가스
  - 알리오 올리오 -> 파스타
  - 김치찌개 -> 찌개
- 메뉴명과 설명 중 하나가 비어 있으면 남은 정보만 보고 판단해.
- 둘 다 비어있으면 메뉴 없음으로 기재해줘
- 설명 문장, 이유, 부가 설명 없이 답만 한 줄로 출력해.

입력:
{input_text}
"""
)

llm = init_chat_model('gpt-4.1-mini', temperature=0)
output_parser = StrOutputParser()

chain = food_prompt | llm | output_parser

In [ ]:
# food_df['sum_name'] = food_df['prompt_input'].apply(
#     lambda x: chain.invoke({"input_text": x}).strip()
# )


# 너무 느려서 중복 제거하고 배치 사용
print(menu_df.info())
food_df = menu_df.drop_duplicates(subset=['name', 'description'], keep='first').reset_index(drop=True)
print(food_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          2008 non-null   object
 1   description   2008 non-null   object
 2   prompt_input  2008 non-null   object
dtypes: object(3)
memory usage: 47.2+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1920 entries, 0 to 1919
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          1920 non-null   object
 1   description   1920 non-null   object
 2   prompt_input  1920 non-null   object
dtypes: object(3)
memory usage: 45.1+ KB
None


In [31]:
# 배치 사용하기
inputs = [{"input_text" : x} for x in menu_df['prompt_input']]

batch_size = 15
all_results = []

for i in range(0, len(inputs), batch_size):
    batch = inputs[i: i+batch_size]
    results = chain.batch(batch)
    all_results.extend(results)

menu_df['sum_name'] = [r.strip() if r else None for r in all_results]

In [32]:
len(menu_df['sum_name'])

2008

In [33]:
# 1. sum_name → food_name 컬럼으로 복사
menu_df['food_name'] = menu_df['sum_name']

# 2. food_code 생성 (FOD0000 ~)
menu_df['food_code'] = [
    f"FOD{str(i).zfill(4)}" for i in range(len(menu_df))
]

In [36]:
menu_df[:3]

,menu_code,restaurant_code,name,price,description,menu_desc,prompted_description,embedding_blobX,embedding,prompt_input,food_name,food_code
0,MEN0000,RES0000,회전초밥,NaN,NaN,회전초밥,회전초밥은 신선한 재료로 만든 다양한 초밥을 회전 컨베이어 벨트 위에서 편리하게 즐...,"[0.01146697998046875, -0.041229248046875, -0.0...",AOA7PADgKL0AYNS7ACAgPACAnzwAAO67AAC8uwBA5zwAII...,menu_name: 회전초밥\n설명:,초밥,FOD0000
1,MEN0001,RES0000,장어초밥,NaN,NaN,장어초밥,"장어초밥은 부드럽고 고소한 장어가 신선한 초밥 위에 올려져 깊은 풍미를 자랑하며, ...","[-0.0072174072265625, -0.06524658203125, -0.00...",AIDsuwCghb0AoBe8AEBdPAAAZD0AAKq8AEBQuwDgET0AYP...,menu_name: 장어초밥\n설명:,초밥,FOD0001
2,MEN0002,RES0001,프레지에 1호 홀케이크,50000.0,NaN,프레지에 1호 홀케이크,"프레지에 1호 홀케이크는 신선한 재료로 정성껏 만든 부드럽고 촉촉한 케이크로, 특별...","[0.0193634033203125, -0.00444793701171875, -0....",AKCePADAkbsA4BS9AACeuwAgrz0AIBK9AOACvQDA9jwAIC...,menu_name: 프레지에 1호 홀케이크\n설명:,케이크,FOD0002


In [35]:
menu_df = menu_df.drop(columns=['sum_name'])

In [ ]:
# food_df의 sum_name 컬럼만 뽑아서 중복값 제거하고 리스트로 만들기
food_list = food_df['sum_name'].drop_duplicates().tolist()
len(food_list)

313

### (1) food_list 임베딩

In [ ]:
from openai import OpenAI
import pandas as pd

client = OpenAI()
menu_embeddings = []

batch_size = 100
for i in range(0, len(food_list), batch_size):
    batch = food_list[i:i+batch_size]
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    menu_embeddings.extend([item.embedding for item in response.data])

menu_emb_df = pd.DataFrame({
    'id': range(len(food_list)),   # 고유 식별자 추가
    'menu_name': food_list,
    'embedding': menu_embeddings
})

In [ ]:
menu_emb_df.to_csv("menu_embedding.csv", index=False)

In [ ]:
# food_list[:100]
# 메뉴없음 제거

food_list_clean = [
    x.strip() for x in food_list
    if x and x.strip() and x.strip() != "메뉴 없음"
]
len(food_list_clean)

312

### (2) 메뉴를 설명하는 문장 생성 후 문단(문장) 임베딩 (csv)추출

In [ ]:
# food_list 한번 더 llm으로 정제
# food_list[:20]

# llm = init_chat_model('gpt-4.1-mini', temperature=0)

# prompt = PromptTemplate.from_template("""
# 너는 음식 메뉴를 표준화하는 분류 봇이다.

# 규칙:
# - 반드시 한 단어 또는 매우 짧은 명사만 출력한다.
# - 절대 문장 형태로 쓰지 마라.
# - 설명, 이유, 조사(입니다, 이다 등) 금지
# - 예:
#   치즈 돈가스 -> 돈가스
#   알리오 올리오 -> 파스타
#   김치찌개 -> 찌개
#   곱창전골 -> 전골
#   곱창 -> 곱창

# 입력:
# {input_text}

# 출력:
# """)

# output_parser = StrOutputParser()

# chain = prompt | llm | output_parser

# # 배치 사용하기
# inputs = [{"input_text" : x} for x in food_list]

# batch_size = 15
# all_results = []

# for i in range(0, len(inputs), batch_size):
#     batch = inputs[i: i+batch_size]
#     results = chain.batch(batch)
#     all_results.extend(results)

# food_list_uq = [r.strip() if r else None for r in all_results]

In [ ]:
# food_list_uq = list(dict.fromkeys(food_list_uq))
# len(food_list_uq)

157

In [ ]:
# food_list_uq[:15]

In [ ]:
# # 캐싱 쓰는 버전
# cache = {}
# all_results = []

# batch_size = 15

# for i in range(0, len(food_list), batch_size):
#     batch_raw = food_list[i:i+batch_size]

#     # 새로 처리할 것만 모음
#     new_inputs = []
#     new_keys = []

#     for x in batch_raw:
#         key = str(x).strip() if x else ""

#         if not key:
#             all_results.append(None)
#         elif key in cache:
#             all_results.append(cache[key])
#         else:
#             new_inputs.append({"input_text": key})
#             new_keys.append(key)
#             all_results.append("__PENDING__")  # 자리 표시

#     # LLM 호출
#     if new_inputs:
#         results = chain.batch(new_inputs)

#         for key, res in zip(new_keys, results):
#             cache[key] = res.strip() if res else None

#     # 결과 채우기
#     for j in range(len(all_results)):
#         if all_results[j] == "__PENDING__":
#             key = new_keys.pop(0)
#             all_results[j] = cache[key]

# food_list_uq = all_results

In [ ]:
# 메뉴 설명 생성
llm = init_chat_model('gpt-4.1-mini', temperature=0)
prompt = PromptTemplate.from_template("""
당신은 음식 메뉴 데이터를 정리하는 전문가다.

주어진 메뉴명을 보고, 해당 음식의 특징, 대표적인 맛, 일반적으로 알려진 영양적 특성을 바탕으로
1~3문장의 짧은 설명 문단을 작성하라.

작성 규칙:
1. 반드시 한국어로 작성한다.
2. 메뉴명에 대한 설명만 작성하고, 불필요한 서론이나 추측성 표현은 쓰지 않는다.
3. 음식의 핵심 재료, 조리 방식, 대표적인 맛의 특징을 자연스럽게 포함한다.
4. 일반적으로 알려진 영양 정보를 포함한다. 예: 단백질이 풍부함, 탄수화물 비중이 높음, 나트륨이 높을 수 있음 등
5. 문장은 1~3문장으로만 작성한다.
6. 출력은 설명 문단만 작성한다. 제목, 번호, 따옴표, 불릿포인트는 금지한다.
7. 메뉴명이 너무 일반적이거나 정보가 불충분한 경우에는, 한국에서 일반적으로 떠올릴 수 있는 대표적인 음식 기준으로 설명한다.
8. 허위로 매우 구체적인 수치(칼로리, g 단위 등)는 만들지 말고, 일반적인 수준에서 설명한다.

예시 1
입력: 김치찌개
출력: 김치찌개는 발효된 김치와 돼지고기 또는 참치 등을 넣어 끓인 한국의 대표적인 찌개로, 얼큰하고 감칠맛 나는 국물 맛이 특징이다. 재료에 따라 단백질을 보충할 수 있지만, 김치와 양념이 들어가 나트륨 함량은 다소 높을 수 있다.

예시 2
입력: 비빔밥
출력: 비빔밥은 밥 위에 다양한 나물, 고기, 계란, 고추장을 곁들여 비벼 먹는 음식으로, 여러 재료가 어우러져 조화로운 맛과 식감을 낸다. 탄수화물, 단백질, 채소를 함께 섭취할 수 있어 비교적 균형 잡힌 한 끼가 될 수 있다.

예시 3
입력: 치킨마요덮밥
출력: 치킨마요덮밥은 밥 위에 닭고기와 마요네즈 소스를 올려 먹는 덮밥으로, 고소하고 짭짤한 맛이 강한 편이다. 탄수화물과 단백질을 함께 섭취할 수 있지만, 소스 사용량에 따라 지방과 열량이 높아질 수 있다.

이제 다음 메뉴에 대해 작성하라.

입력: {menu_name}
출력:
""")

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

In [ ]:
# 배치 사용하기
# langsmith 한도 초과로 기능 끔
import os
os.environ["LANGCHAIN_TRACING"] = "false"

inputs = [{"menu_name" : x} for x in food_list_clean]

batch_size = 20
all_results = []

for i in range(0, len(inputs), batch_size):
    batch = inputs[i: i+batch_size]
    results = chain.batch(batch)
    all_results.extend(results)

# 문단 리스트 생성
desc_list = [r.strip() for r in all_results if r]

In [ ]:
from openai import OpenAI
import pandas as pd

client = OpenAI()
desc_embeddings = []

batch_size = 100
for i in range(0, len(desc_list), batch_size):
    batch = food_list_clean[i:i+batch_size]
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    desc_embeddings.extend([item.embedding for item in response.data])

In [ ]:
desc_list[1]

'케이크는 밀가루, 설탕, 달걀, 버터 등을 주재료로 하여 오븐에 구워 만든 달콤한 디저트로, 부드럽고 촉촉한 식감이 특징이다. 주로 당분과 탄수화물 함량이 높아 에너지원이 되지만, 지방과 칼로리도 비교적 높을 수 있다.'

In [ ]:
len(food_list_clean), len(desc_list), len(desc_embeddings)

(312, 312, 312)

In [ ]:
menu_desc_emb_df = pd.DataFrame({
    'id': range(len(food_list_clean)),   # 고유 식별자 추가
    'menu_name': food_list_clean[:len(desc_list)],
    'description': desc_list,
    'embedding': desc_embeddings
})

In [ ]:
menu_desc_emb_df[:1]

,id,menu_name,description,embedding
0,0,초밥,"초밥은 식초로 간을 한 밥 위에 신선한 생선이나 해산물을 올려 만든 일본식 요리로,...","[0.0058135986328125, -0.05877685546875, -0.031..."


In [ ]:
menu_desc_emb_df.to_csv("C:/project_skn/3rd_project/database/sql/db_csv/food.csv", index=False)

## BLOB 임베딩

In [ ]:
import pandas as pd
import numpy as np
import ast

# 1. CSV 읽기
df = pd.read_csv("food_embedding_blobX.csv")

# 2. embedding 컬럼을 문자열 -> 리스트로 변환
def parse_embedding(x):
    if pd.isna(x):
        return None
    if isinstance(x, list):
        return x
    return ast.literal_eval(x)

# 3. 리스트 -> numpy bytes(BLOB) 변환
def embedding_to_blob(x):
    if x is None:
        return None
    arr = np.array(x, dtype=np.float32)
    return arr.tobytes()

# BLOB 컬럼 생성
df["blob_embedding"] = df["embedding"].apply(parse_embedding).apply(embedding_to_blob)

# 🔥 컬럼명 변경
df.rename(columns={"embedding": "embedding_blobX"}, inplace=True)
df.rename(columns={"blob_embedding": "embedding"}, inplace=True)

print(df[["embedding_blobX", "embedding"]].head())

                                     embedding_blobX  \
0  [0.0058135986328125, -0.05877685546875, -0.031...   
1  [0.0086669921875, -0.055389404296875, -0.07781...   
2  [-0.047698974609375, -0.00655364990234375, -0....   
3  [-0.018157958984375, -0.026763916015625, -0.03...   
4  [-0.01326751708984375, -0.0089874267578125, -0...   

                                           embedding  
0  b'\x00\x80\xbe;\x00\xc0p\xbd\x00\xe0\x01\xbd\x...  
1  b'\x00\x00\x0e<\x00\xe0b\xbd\x00`\x9f\xbd\x00\...  
2  b'\x00`C\xbd\x00\xc0\xd6\xbb\x00\xa0\xc5\xbc\x...  
3  b'\x00\xc0\x94\xbc\x00@\xdb\xbc\x00\xe0\x1c\xb...  
4  b'\x00`Y\xbc\x00@\x13\xbc\x00`\x13\xbd\x00\xe0...  


In [ ]:
# 1. id → food_code로 컬럼명 변경
df.rename(columns={"id": "food_code"}, inplace=True)

# 2. 값 재생성 (FOD0000, FOD0001, ...)
df["food_code"] = [f"FOD{str(i).zfill(4)}" for i in range(len(df))]

print(df[["food_code"]].head())

  food_code
0   FOD0000
1   FOD0001
2   FOD0002
3   FOD0003
4   FOD0004


In [ ]:
df.to_csv("food.csv", index=False)